# NewsLens — Multi-Document News Summarization Experiment
### Step-by-Step Interactive Pipeline Demonstration

This notebook demonstrates each stage of the NewsLens pipeline:
1. Ingesting and validating news articles
2. Text preprocessing and named entity recognition
3. Topic modeling using Non-Negative Matrix Factorization (NMF)
4. Semantic distance computation and Agglomerative event clustering
5. Extractive summarization with Maximal Marginal Relevance (MMR)
6. Abstractive and Hybrid summarization with source attribution
7. ROUGE metric and cluster purity evaluation

In [ ]:
import sys
from pathlib import Path
# Ensure project root is in sys.path
sys.path.append(str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))

import pandas as pd
import numpy as np
import config
from src.data_loader import load_dataset, validate_dataset, clean_dataset, get_dataset_summary
from src.preprocessing import preprocess_dataframe, split_sentences
from src.topic_modeling import TopicModeler
from src.similarity import ArticleSimilarity
from src.event_clustering import EventClusterer
from src.extractive_summarizer import ExtractiveSummarizer
from src.abstractive_summarizer import AbstractiveSummarizer
from src.hybrid_summarizer import HybridSummarizer
from src.evaluation import evaluate_summary, evaluate_clustering

print('NewsLens modules imported successfully!')

## 1. Data Ingestion & Cleaning

In [ ]:
df_raw = load_dataset(config.SAMPLE_DATA_PATH, file_type='csv')
is_valid, errors = validate_dataset(df_raw)
print(f'Dataset schema valid: {is_valid}')

df_clean, stats = clean_dataset(df_raw)
print(f'Cleaned dataset: {len(df_clean)} articles (filtered {stats["duplicates_removed"]} duplicates)')
df_clean[['article_id', 'title', 'source']].head(4)

## 2. Text Preprocessing & Entity Recognition

In [ ]:
df_proc = preprocess_dataframe(df_clean)
print('Sample Extracted Entities for Article 1:')
print(df_proc.loc[0, 'entities'])
df_proc[['title', 'tokens', 'sentences']].head(2)

## 3. Topic Modeling with NMF

In [ ]:
modeler = TopicModeler(n_topics=4, method='nmf')
modeler.fit_transform(df_proc['processed_text'].tolist())
keywords = modeler.get_topic_keywords(top_n=5)
for t_id, kw in keywords.items():
    print(f'Topic {t_id + 1}: {", ".join(kw)}')

dom_topics, conf, labels = modeler.assign_topics(df_proc['processed_text'].tolist())
df_proc['dominant_topic'] = labels

## 4. Semantic Similarity & Event Clustering

In [ ]:
clusterer = EventClusterer(distance_threshold=0.75)
cluster_ids = clusterer.fit_predict(
    df_proc['processed_text'].tolist(),
    titles=df_proc['title'].tolist(),
    article_ids=df_proc['article_id'].tolist()
)
df_proc['cluster_id'] = cluster_ids

clusters = clusterer.get_cluster_summary()
print(f'Total candidate events discovered: {len(clusters)}')
for c in clusters:
    print(f'\n[Event {c["event_id"]}] ({c["article_count"]} articles): {c["representative_title"]}')
    for t in c['titles']:
        print(f'  - {t}')

## 5. Multi-Document Summarization Studio

In [ ]:
# Summarize Event 1 (Mumbai Floods)
event_1_articles = df_clean.iloc[clusters[0]['article_indices']].to_dict('records')

# A. Extractive Summarization (MMR)
ext_summarizer = ExtractiveSummarizer()
ext_res = ext_summarizer.summarize_articles(event_1_articles, num_sentences=2)
print('--- EXTRACTIVE SUMMARY ---')
print(ext_res['summary_text'])

# B. Hybrid Summarization with Source Provenance
hybrid_summarizer = HybridSummarizer()
hybrid_res = hybrid_summarizer.summarize_event(event_1_articles, num_extracted_sentences=2)
print('\n--- HYBRID SUMMARY ---')
print(hybrid_res['hybrid_summary'])

print('\n--- SUPPORTING PASSAGES & PROVENANCE ---')
for p in hybrid_res['supporting_passages']:
    print(f'[{p["source"]}] {p["passage"]}')

## 6. Evaluation & Metrics

In [ ]:
# ROUGE evaluation against ground-truth reference summary
ref_summary = event_1_articles[0].get('reference_summary', '')
rouge_eval = evaluate_summary(hybrid_res['hybrid_summary'], ref_summary)

print('ROUGE Evaluation Results:')
for metric_name, scores in rouge_eval['metrics'].items():
    print(f'{metric_name} -> F1: {scores["f1"]}, Precision: {scores["precision"]}, Recall: {scores["recall"]}')

# Clustering quality evaluation
if 'event_id' in df_clean.columns:
    c_eval = evaluate_clustering(df_clean['event_id'].tolist(), df_proc['cluster_id'].tolist())
    print('\nClustering Evaluation:')
    print(f'Adjusted Rand Index (ARI): {c_eval["metrics"].get("Adjusted_Rand_Index_ARI")}')
    print(f'Normalized Mutual Info (NMI): {c_eval["metrics"].get("Normalized_Mutual_Info_NMI")}')